In [1]:
import pandas as pd
import sqlite3
import os

RAW_DATA = r"C:\Users\ASUS ROG\last-mile-delivery-intelligence\data\raw"
DB_PATH  = r"C:\Users\ASUS ROG\last-mile-delivery-intelligence\data\supply_chain.db"

print("Libraries imported successfully")
print(f"Raw folder exists: {os.path.exists(RAW_DATA)}")

Libraries imported successfully
Raw folder exists: True


In [2]:
# Preview Olist orders
orders_preview = pd.read_csv(f"{RAW_DATA}\\olist_orders_dataset.csv")
print(f"Orders shape: {orders_preview.shape}")
print(f"Columns: {list(orders_preview.columns)}")
orders_preview.head(3)

Orders shape: (99441, 8)
Columns: ['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date']


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00


In [3]:
# Preview shipping dataset - confirm target column exists
shipping_preview = pd.read_csv(f"{RAW_DATA}\\shipping_dataset.csv")
print(f"Shipping shape: {shipping_preview.shape}")
print(f"Columns: {list(shipping_preview.columns)}")
shipping_preview.head(3)

Shipping shape: (10999, 12)
Columns: ['ID', 'Warehouse_block', 'Mode_of_Shipment', 'Customer_care_calls', 'Customer_rating', 'Cost_of_the_Product', 'Prior_purchases', 'Product_importance', 'Gender', 'Discount_offered', 'Weight_in_gms', 'Reached.on.Time_Y.N']


,ID,Warehouse_block,Mode_of_Shipment,Customer_care_calls,Customer_rating,Cost_of_the_Product,Prior_purchases,Product_importance,Gender,Discount_offered,Weight_in_gms,Reached.on.Time_Y.N
0,1,D,Flight,4,2,177,3,low,F,44,1233,1
1,2,F,Flight,4,5,216,2,low,M,59,3088,1
2,3,A,Flight,2,2,183,4,low,M,48,3374,1


In [4]:
# Connect to SQLite database
conn = sqlite3.connect(DB_PATH)
print(f"Connected to database")

# Define all Olist files to load
olist_tables = {
    'orders':      'olist_orders_dataset.csv',
    'order_items': 'olist_order_items_dataset.csv',
    'customers':   'olist_customers_dataset.csv',
    'reviews':     'olist_order_reviews_dataset.csv',
    'sellers':     'olist_sellers_dataset.csv',
    'products':    'olist_products_dataset.csv',
    'payments':    'olist_order_payments_dataset.csv',
    'geolocation': 'olist_geolocation_dataset.csv',
}

# Load each file into SQLite
for table_name, filename in olist_tables.items():
    filepath = os.path.join(RAW_DATA, filename)
    df = pd.read_csv(filepath)
    df.to_sql(table_name, conn, if_exists='replace', index=False)
    print(f"✅ Loaded '{table_name}': {len(df):,} rows")

print("\nAll Olist tables loaded!")

Connected to database
✅ Loaded 'orders': 99,441 rows
✅ Loaded 'order_items': 112,650 rows
✅ Loaded 'customers': 99,441 rows
✅ Loaded 'reviews': 99,224 rows
✅ Loaded 'sellers': 3,095 rows
✅ Loaded 'products': 32,951 rows
✅ Loaded 'payments': 103,886 rows
✅ Loaded 'geolocation': 1,000,163 rows

All Olist tables loaded!


In [5]:
# Load shipping dataset with correct target column rename
shipping_df = pd.read_csv(f"{RAW_DATA}\\shipping_dataset.csv")

# Rename target column to clean name
shipping_df = shipping_df.rename(
    columns={'Reached.on.Time_Y.N': 'on_time'}
)

# Load into SQLite
shipping_df.to_sql('shipments', conn, if_exists='replace', index=False)

print(f"✅ Loaded 'shipments': {len(shipping_df):,} rows")
print(f"\nTarget variable distribution:")
print(shipping_df['on_time'].value_counts())
print("\n1 = On time  |  0 = Late/Failed")

✅ Loaded 'shipments': 10,999 rows

Target variable distribution:
on_time
1    6563
0    4436
Name: count, dtype: int64

1 = On time  |  0 = Late/Failed


In [6]:
# Verify all tables loaded correctly
cursor = conn.cursor()
cursor.execute("SELECT name FROM sqlite_master WHERE type='table' ORDER BY name;")
tables = cursor.fetchall()

print("=" * 40)
print("Tables in supply_chain.db:")
print("=" * 40)
for table in tables:
    table_name = table[0]
    cursor.execute(f"SELECT COUNT(*) FROM {table_name}")
    count = cursor.fetchone()[0]
    print(f"  {table_name:<20} {count:>8,} rows")

conn.close()
print("\n✅ Database ready. Connection closed.")

Tables in supply_chain.db:
  customers              99,441 rows
  geolocation          1,000,163 rows
  order_items           112,650 rows
  orders                 99,441 rows
  payments              103,886 rows
  products               32,951 rows
  reviews                99,224 rows
  sellers                 3,095 rows
  shipments              10,999 rows

✅ Database ready. Connection closed.
